In [17]:
# Import data
import pandas as pd
salesdata = pd.read_csv('pcrdata.csv')

In [18]:
# fit a linear model
import statsmodels.formula.api as smf

# smf.ols() fit a linear regression model
predsales = smf.ols('SALES~AD+PRO+SALEXP+PROPRE', data = salesdata).fit()
predsales.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  SALES   R-squared:                       0.905
Model:                            OLS   Adj. R-squared:                  0.902
Method:                 Least Squares   F-statistic:                     326.8
Date:                Tue, 22 Sep 2026   Prob (F-statistic):           2.64e-69
Time:                        13:18:08   Log-Likelihood:                -229.42
No. Observations:                 143   AIC:                             468.8
Df Residuals:                     138   BIC:                             483.7
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      5.7257      0.981      5.839      0.000       3.787       7.665
AD             1.1745      0.350      3.358      0.001       0.483       1.866
PRO            4.6471      0.304     15.272      0.000       4.045       5.249
SALEXP        22.6275      0.781     28.977      0.000      21.083      24.171
PROPRE         0.0793      0.313      0.253      0.800      -0.540       0.698
==============================================================================
Omnibus:                        2.422   Durbin-Watson:                   2.138
Prob(Omnibus):                  0.298   Jarque-Bera (JB):                2.213
Skew:                           0.214   Prob(JB):                        0.331
Kurtosis:                       2.566   Cond. No.                         25.0
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [19]:
# checking for Multicollinearity
from patsy import dmatrices
from statsmodels.stats.outliers_influence import variance_inflation_factor
y, X = dmatrices('SALES~AD+PRO+SALEXP+ADPRE+PROPRE', data=salesdata, return_type = "dataframe")
vif = pd.Series([variance_inflation_factor(X.values, i) for i in range(X.shape[1])], index=X.columns)
vif

Intercept    4226.760949
AD             36.159771
PRO            31.846727
SALEXP          1.076284
ADPRE          24.781948
PROPRE         42.346468
dtype: float64

In [20]:
# PCA in Python
from sklearn.decomposition import PCA
from sklearn.preprocessing import scale

salesdata2=salesdata.drop(['SRNO', 'SALES'], axis=1)
standardisedX = scale(salesdata2)
X = pd.DataFrame(standardisedX, index=salesdata2.index,columns=salesdata2.columns)
pca = PCA().fit(X)

In [21]:
# summary of PCA
import numpy as np
names = ["PC"+str(i) for i in range(1,6)]

SD = list(np.std(pca.transform(X), axis = 0))
VarProp = list(pca.explained_variance_ratio_)
CumProp = [np.sum(pca.explained_variance_ratio_[:i]) for i in range(1,6)]

summary = pd.DataFrame(list(zip(SD, VarProp, CumProp)), index=names, columns=['Standard Deviation', 'Proportion of Variance', 'Cumulative Proportion'])
summary

,Standard Deviation,Proportion of Variance,Cumulative Proportion
PC1,1.301556,0.338810,0.338810
PC2,1.131848,0.256216,0.595026
PC3,1.070535,0.229209,0.824235
PC4,0.933433,0.174259,0.998494
PC5,0.086770,0.001506,1.000000


In [22]:
%pip install hoggorm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [23]:
# import libarary hoggorm for PCR in Python
import hoggorm
pcmodel = hoggorm.pcr.nipalsPCR(standardisedX, y)
salesdata['pcrpred'] = pcmodel.Y_predCal()[3]
salesdata.head()

,SRNO,SALES,AD,PRO,SALEXP,ADPRE,PROPRE,pcrpred
0,1,20.11,1.98,0.9,0.31,2.02,0.0,21.290481
1,2,15.10,1.94,0.0,0.30,1.99,1.0,18.169735
2,3,18.68,2.20,0.8,0.35,1.93,0.0,21.271485
3,4,16.05,2.00,0.0,0.35,2.20,0.8,17.621109
4,5,21.30,1.69,1.3,0.30,2.00,0.0,22.979205


In [24]:
import sys
print(sys.version)

3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]


In [25]:
salesdata_test = pd.read_csv('pcrdata_test.csv')

In [26]:
# Getting RMSE of PCR model 
salesdata_test2=salesdata_test.drop(['SRNO','SALES'], axis=1)
standardisedX2 = scale(salesdata_test2)
salesdata_test['pcrpredict'] =  pcmodel.Y_predict(standardisedX2, numComp=3) 
salesdata_test['pcrres'] = salesdata_test['SALES'] - salesdata_test['pcrpredict']
import math
import statistics
RMSE_pcr = math.sqrt(statistics.mean(salesdata_test['pcrres']**2))

In [30]:
salesdata_test['lmpred']= predsales.predict(salesdata_test)
salesdata_test['lmres'] = salesdata_test['SALES'] - salesdata_test['lmpred']

RMSE_lm = math.sqrt(statistics.mean(salesdata_test['lmres']**2))

In [32]:
# view data
salesdata_test.head()

,SRNO,SALES,AD,PRO,SALEXP,ADPRE,PROPRE,pcrpredict,pcrres,lmpred,lmres
0,1,28.93,2.75,1.00,0.72,1.97,0.02,22.564800,6.365200,29.896044,-0.966044
1,2,25.96,1.73,1.06,0.89,2.77,0.02,21.752214,4.207786,32.823556,-6.863556
2,3,31.25,2.19,1.26,0.79,1.22,0.42,26.662886,4.587114,32.062211,-0.812211
3,4,25.05,1.82,1.45,0.83,2.23,0.15,24.720419,0.329581,33.394292,-8.344292
4,5,27.32,2.38,1.01,0.74,1.01,0.07,25.930267,1.389733,29.964465,-2.644465


In [36]:
RMSE_lm

7.218597741092888

In [38]:
RMSE_pcr

3.0135616853589267

In [ ]:
# RMSE using PCR is less than RMSE using linear regression, we may conclude that PCR mode predicts SALES better than linear regression model when multicollinearity esists among the independent variables.
